In [0]:
%sql
--Reporte de Calidad de datos

WITH total AS (
  SELECT
    COUNT(*) AS total_registros 
  FROM estadisticas_clean
),
problemas_calidad AS (

SELECT
  COUNT(CASE WHEN precio IS NULL OR precio <=0 THEN 1 END) AS precio_invalido,

  COUNT(CASE WHEN metros_cuadrados_totales IS NULL OR metros_cuadrados_totales <=0 THEN 1 END) AS m2_totales_invalido,

  COUNT(CASE WHEN antiguedad = '999' OR antiguedad= 999 THEN 1 END) AS antiguedad_invalido,

  COUNT(CASE WHEN ambientes IS NULL OR ambientes = 0 THEN 1 END) AS ambientes_invalido,

  COUNT(CASE WHEN moneda IS NULL OR moneda = '' THEN 1 END) AS moneda_invalido,

  COUNT(CASE WHEN tipo_de_operacion IS NULL OR tipo_de_operacion = '' THEN 1 END) AS tipo_de_operacion_invalido
FROM estadisticas_clean
)
SELECT
  t.total_registros,

  p.precio_invalido,
  ROUND(p.precio_invalido * 100.0 / t.total_registros,2) AS 
  pct_precio_invalido,

  p.m2_totales_invalido,
  ROUND(p.m2_totales_invalido * 100.0 / t.total_registros,2) AS pct_m2_totales_invalido,

  p.antiguedad_invalido,
  ROUND(p.antiguedad_invalido * 100.0 / t.total_registros,2) AS pct_antiguedad_invalido,

  p.ambientes_invalido,
  ROUND(p.ambientes_invalido * 100.0 / t.total_registros,2) AS pct_ambientes_invalido,

  p.moneda_invalido,
  ROUND(p.moneda_invalido * 100.0 / t.total_registros,2) AS pcte_moneda_invalido,

  p.tipo_de_operacion_invalido,
  ROUND(p.tipo_de_operacion_invalido * 100.0 / t.total_registros,2) AS pct_tipo_de_operacion_invalido
FROM total t, problemas_calidad p


/*
El dataset presenta calidad aceptable en la mayoría de las variables, excepto en antigüedad, que muestra un problema crítico debido al uso masivo de valores placeholder.
Además, se detectan inconsistencias moderadas en superficies y ambientes, así como pequeñas irregularidades en variables categóricas como moneda y tipo de operación.
*/


In [0]:
%sql
--Analisis de valores duplicados
WITH duplicados AS (
  SELECT
    url,
    precio,
    COUNT(*) AS cantidad
  FROM bootcamp_matias.raw.propiedades_bronze
  GROUP BY
    url,precio
  HAVING COUNT(*) > 1
)
SELECT
  COUNT(*) AS grupos_duplicados,
  SUM(cantidad) AS total_duplicados,
  SUM(cantidad - 1) AS extras_por_duplicacion
FROM duplicados



In [0]:
%sql
--Ver datos duplicados
WITH duplicados AS (
  SELECT
    url,
    precio,
    COUNT(*) AS cantidad
  FROM bootcamp_matias.raw.propiedades_bronze
  GROUP BY
    url,precio
  HAVING COUNT(*) > 1
)
SELECT * FROM duplicados LIMIT 20

In [0]:
%sql
--Posible outliers en precio por moneda
WITH percentiles AS (
  SELECT
    moneda,
    PERCENTILE(precio, 0.01) AS p01,
    PERCENTILE(precio, 0.99) AS  p99
  FROM estadisticas_clean
  WHERE precio > 0
  GROUP BY moneda
)
SELECT
  e.moneda,
  'Muy Bajo <p01' AS tipo_outliers,
  COUNT(*) AS cantidad
FROM estadisticas_clean e
JOIN percentiles pc
  ON pc.moneda = e.moneda
WHERE 
  e.precio < pc.p01 AND precio > 0
GROUP BY 
  e.moneda

UNION ALL

SELECT
  e.moneda,
  'Muy Bajo > p99' AS tipo_outliers,
  COUNT(*) AS cantidad
FROM estadisticas_clean e
JOIN percentiles pc
  ON pc.moneda = e.moneda
WHERE 
  e.precio > pc.p99 AND precio > 0
GROUP BY 
  e.moneda
ORDER BY cantidad  DESC
